# Phase 5: KPI Calculation and Final Tableau Load Preparation

This notebook prepares the final dataset for Tableau dashboarding
The goal is to create a clean, aggregated, analysis-ready dataset
The dataset should:
- Contain key KPIs
- Support filtering and drill-down
- Be optimized for visualization

## KPI Definitions
- Total Revenue = sum of all order values per customer
- Average Order Value = total revenue / total orders
- Recency = number of days since last purchase

## Final Output
Dataset created for dashboard consumption
Contains all required KPIs
Supports filtering and segmentation

## What Your Dashboard Will Use
This dataset should support:
- Customer segmentation
- Churn analysis
- Revenue analysis
- Category insights
- Delivery performance

This notebook is the bridge between your Python analysis and your Tableau dashboard. The goal is to calculate the final metrics and export a "wide" dataset that Tableau can ingest perfectly without requiring complex joins inside the visualization tool.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('../data/processed/final_dataset.csv')

In [ ]:
df['total_order_value'] = df['price'] + df['freight_value']

In [ ]:
df['freight_ratio'] = df['freight_value'] / df['total_order_value']

In [ ]:
df['is_late_delivery'] = np.where(df['order_delivered_customer_date'] > df['order_estimated_delivery_date'], 1, 0)

In [ ]:
customer_kpis = df.groupby('customer_unique_id').agg(
    Customer_Lifetime_Value=('total_payment_value', 'sum'),
    Customer_Order_Count=('order_id', 'nunique'),
    Customer_Average_Order_Value=('total_payment_value', 'mean')
).reset_index()

In [ ]:
df = df.merge(customer_kpis, on='customer_unique_id', how='left')

In [ ]:
seller_kpis = df.groupby('seller_id').agg(
    Seller_Total_Revenue=('price', 'sum'),
    Seller_Average_Review_Score=('review_score', 'mean'),
    Seller_Late_Delivery_Rate=('is_late_delivery', lambda x: x.sum() / x.count() if x.count() > 0 else 0)
).reset_index()

In [ ]:
df = df.merge(seller_kpis, on='seller_id', how='left')

In [ ]:
# Drop redundant columns that are no longer needed for Tableau
columns_to_drop = ['customer_id', 'order_item_id', 'payment_sequential', 'payment_installments']
df.drop(columns=[col for col in columns_to_drop if col in df.columns], inplace=True)

In [ ]:
df.to_csv('../data/processed/tableau_export.csv', index=False)

Dataset successfully prepped and exported as tableau_export.csv. Ready for Phase 6: Tableau Dashboarding.